# Algorithm 5 test

In [2]:
from utils import *

q = 2             # Base field characteristic
m = 4             # Field extension degree 
n = 7             # Total code length 
k = 3             # Dimension of the code
r = 2             # Target rank weight of the error vector

n_minus_k = n - k 
Fq = GF(q)
Fqm = GF(q**m, name='z')
V_Fq = Fqm.vector_space()

print(f"Parameters: q={q}, m={m}, n={n}, k={k}, r={r}\n")

# Generate a random Parity-Check matrix H with full row rank
while True:
    H = matrix(Fqm, n_minus_k, n, [Fqm.random_element() for _ in range(n_minus_k * n)])
    if H.rank() == n_minus_k:
        break

# Generate a secret error vector with exact rank r
field_basis = [Fqm.gen()**i for i in range(m)]
support_basis = random.sample(field_basis, r) # Sample r linearly independent elements

e_list = []
for _ in range(n):
    coeffs = [Fq.random_element() for _ in range(r)]
    entry = sum(int(coeffs[i]) * support_basis[i] for i in range(r))
    e_list.append(entry)
e_secret = vector(Fqm, e_list)

E_matrix = matrix(Fq, [V_Fq(el) for el in e_secret]).transpose()
assert E_matrix.rank() == r, "Error vector rank mismatch."

print(f"Generated Secret Vector e:\n{e_secret}")
print(f"Underlying Support Matrix E (dimensions {m}x{n}):\n{E_matrix}\n")

# Compute Syndrome
s = H * e_secret

num_erasures = 2
J = sorted(random.sample(range(n), num_erasures))

a_J = []
for j in J:
    # Extract the j-th column from the base field matrix E
    column_vector_Fq = E_matrix.column(j)
    
    # Reconstruct the extension field element from its Fq column coordinates
    field_element_Fqm = sum(column_vector_Fq[i] * field_basis[i] for i in range(m))
    a_J.append(Fqm(field_element_Fqm))

print("=== [2] Extracting Correct Column Erasure Hints from Matrix E ===")
print(f"Selected Column Indices (J): {J}")
for idx, j in enumerate(J):
    print(f"  -> Column {j} from matrix E: {list(E_matrix.column(j))} maps to field element: {a_J[idx]}")
print(f"\nFinal a_J array passed to decoder: {a_J}\n")

# =====================================================================
# 3. RUN DECODER AND VERIFY
# =====================================================================
print("=== [3] Running Full-Entry AP Decoder Module ===")
candidate_list = full_entry_ap_decoder_module(H, s, J, a_J, q, m, n, k)
print(f"Number of candidates returned: {len(candidate_list)}")

# Final Verification
success = False
for idx, candidate in enumerate(candidate_list):
    syndrome_check = (H * candidate == s)
    cand_matrix = matrix(Fq, [V_Fq(el) for el in candidate]).transpose()
    rank_check = (cand_matrix.rank() == r)
    
    if syndrome_check and rank_check:
        print(f"[+] Candidate #{idx+1} passes validation filters.")
        if candidate == e_secret:
            print("[++++] SUCCESS: Reconstructed vector matches the secret error exactly!")
            success = True
            break

if not success:
    print("[-] FAILURE: Reconstructed target vector could not be recovered.")

Parameters: q=2, m=4, n=7, k=3, r=2

Generated Secret Vector e:
(0, z^2, z, z^2, z^2, z^2 + z, z^2 + z)
Underlying Support Matrix E (dimensions 4x7):
[0 0 0 0 0 0 0]
[0 0 1 0 0 1 1]
[0 1 0 1 1 1 1]
[0 0 0 0 0 0 0]

=== [2] Extracting Correct Column Erasure Hints from Matrix E ===
Selected Column Indices (J): [1, 3]
  -> Column 1 from matrix E: [0, 0, 1, 0] maps to field element: z^2
  -> Column 3 from matrix E: [0, 0, 1, 0] maps to field element: z^2

Final a_J array passed to decoder: [z^2, z^2]

=== [3] Running Full-Entry AP Decoder Module ===
Number of candidates returned: 6
[+] Candidate #5 passes validation filters.
[++++] SUCCESS: Reconstructed vector matches the secret error exactly!


/usr/lib/python3/dist-packages/sage/categories/fields.py:533: DeprecationWarning: The default value for map will be changing to True.  To keep the current behavior, explicitly pass map=False.
See https://trac.sagemath.org/28481 for details.
  return self.free_module(*args, **kwds)


# Algorithm 2 test

In [3]:
from utils import *
import random

q = 2  # Base field size
m = 4  # Extension degree
n = 7  # Code length
k = 3  # Code dimension
d = 3  # Candidate super-support space dimension

Fq = GF(q)
Fqm.<z> = GF(q^m)
V_space = Fqm.vector_space(map=False)


H = matrix(Fqm, n - k, n, lambda i, j: Fqm.random_element())

support_basis = []
while len(support_basis) < d:
    elem = Fqm.random_element()
    if elem != 0 and V_space(elem) not in VectorSpace(Fq, m).subspace([V_space(b) for b in support_basis]):
        support_basis.append(elem)

mock_error_coords = []
for j in range(n):
    coord_element = sum(Fq(random.randint(0, q-1)) * support_basis[lambda_] for lambda_ in range(d))
    mock_error_coords.append(coord_element)
e_mock = vector(Fqm, mock_error_coords)

s = H * e_mock

true_rank_matrix = matrix(Fq, [V_space(el) for el in e_mock]).transpose()
r = true_rank_matrix.rank()

print(f"Parity-Check Matrix H ({H.nrows()}x{H.ncols()}):\n{H}\n")
print(f"Syndrome Vector s    : {s}")
print(f"Secret Target e_mock : {e_mock}")
print(f"Target Rank Weight r : {r}\n")

T = [(0, 1), (3, 2)]
a_T = {}
for (j, ell) in T:
    true_bit_val = V_space(e_mock[j])[ell]
    a_T[(j, ell)] = true_bit_val

for (j, ell) in T:
    print(f"  Side-Information Hint -> Column {j}, Bit Position {ell} = {a_T[(j, ell)]}")
print()


E_F, equations, beta_matrix, R_poly = grs_scalar_erasures(
    q, m, n, d, support_basis, Fqm, H, s, T, a_T, r, B_rec=1000
)

valid_solutions = verify_decoder_candidates(E_F, H, s, r, Fqm)

if e_mock in valid_solutions:
    print("The secret target error vector was successfully recovered and verified!")
else:
    print("The secret target vector was missed by the decoder or filtering process.")

Parity-Check Matrix H (4x7):
[              z^3             z + 1       z^2 + z + 1         z^3 + z^2                 z z^3 + z^2 + z + 1           z^2 + 1]
[                1     z^3 + z^2 + z                 z               z^2           z^3 + z                 1         z^3 + z^2]
[      z^2 + z + 1                 0     z^3 + z^2 + z z^3 + z^2 + z + 1       z^3 + z + 1               z^2         z^3 + z^2]
[          z^3 + 1           z^3 + z           z^3 + 1             z + 1               z^3         z^3 + z^2     z^3 + z^2 + 1]

Syndrome Vector s    : (z^3 + z, z^2 + z, z^2 + 1, z^3 + z + 1)
Secret Target e_mock : (z^3 + z^2 + z, z^3, z^3, 0, z^3 + z^2 + 1, z + 1, z^3)
Target Rank Weight r : 3

  Side-Information Hint -> Column 0, Bit Position 1 = 1
  Side-Information Hint -> Column 3, Bit Position 2 = 0

The secret target error vector was successfully recovered and verified!


# Algorithm 3 test

In [8]:
import random
from utils import *

q = 2  # Base field size
m = 6  # Extension field degree
n = 7  # Total code length
k = 3  # Code dimension
r = 2  # Target rank weight of the error vector

Fq = GF(q)
Fqm.<z> = GF(q^m)
V_space = Fqm.vector_space(map=False)

H = matrix(Fqm, n - k, n, lambda i, j: Fqm.random_element())

# Build error support space
true_support = []
while len(true_support) < r:
    elem = Fqm.random_element()
    if elem != 0 and V_space(elem) not in VectorSpace(Fq, m).subspace([V_space(b) for b in true_support]):
        true_support.append(elem)

# Synthesize coordinates mapping to true_support
mock_error_coords = []
for j in range(n):
    coord_element = sum(Fq(random.randint(0, q-1)) * true_support[i] for i in range(r))
    mock_error_coords.append(coord_element)
e_mock = vector(Fqm, mock_error_coords)

s = H * e_mock
E_matrix = matrix(Fq, [V_space(el) for el in e_mock]).transpose()

print(f"Secret Target Vector e:\n{e_mock}\n")
print(f"Base Field Matrix E:\n{E_matrix}\n")
print(f"Syndrome Vector s: {s}\n")


num_erasures = 2  
J = sorted(random.sample(range(n), num_erasures))
a_J = [e_mock[j] for j in J]

print(f"Column Indices J : {J}")
print(f"Field Elements a_J : {a_J}\n")


# Execute the module with verbose printing enabled via loop tracking
print("[*] Running ap_erasure_decoder_module. Please wait for loop iterations...")
verified_candidates = ap_erasure_decoder_module(
    q=q, m=m, n=n, k=k, r=r, J=J, a_J=a_J, H=H, s=s, B_sup=None, epsilon=0.01, B_rec=128
)

print(f"\nTotal verified candidates found: {len(verified_candidates)}")

found_secret = False
for index, candidate in enumerate(verified_candidates):
    if candidate == e_mock:
        print(f"[SUCCESS] Candidate index {index} recovered the target e_mock exactly!")
        found_secret = True
        print(candidate)
        print(e_mock)
        break

if not found_secret:
    print("[-] Secret vector was not recovered.")

Secret Target Vector e:
(z^5 + z^4 + z^2, 0, 0, z^5 + z^4 + z^2, z^3, z^5 + z^4 + z^2, z^5 + z^4 + z^3 + z^2)

Base Field Matrix E:
[0 0 0 0 0 0 0]
[0 0 0 0 0 0 0]
[1 0 0 1 0 1 1]
[0 0 0 0 1 0 1]
[1 0 0 1 0 1 1]
[1 0 0 1 0 1 1]

Syndrome Vector s: (z^2 + z, z^5 + z^3 + z, z^4 + z^3 + z + 1, z^2 + 1)

Column Indices J : [1, 3]
Field Elements a_J : [0, z^5 + z^4 + z^2]

[*] Running ap_erasure_decoder_module. Please wait for loop iterations...

Total verified candidates found: 1
[SUCCESS] Candidate index 0 recovered the target e_mock exactly!
(z^5 + z^4 + z^2, 0, 0, z^5 + z^4 + z^2, z^3, z^5 + z^4 + z^2, z^5 + z^4 + z^3 + z^2)
(z^5 + z^4 + z^2, 0, 0, z^5 + z^4 + z^2, z^3, z^5 + z^4 + z^2, z^5 + z^4 + z^3 + z^2)


# Algorithm 4 test

In [13]:
import random
from utils import *


q = 2  # Base field size
m = 4  # Extension field degree
n = 7  # Total code length
k = 3  # Code dimension
r = 2  # Target rank weight of the error vector

Fq = GF(q)
Fqm.<z> = GF(q^m)
V_space = Fqm.vector_space(map=False)


H_mat = matrix(Fqm, n - k, n, lambda i, j: Fqm.random_element())

true_support = []
while len(true_support) < r:
    elem = Fqm.random_element()
    if elem != 0 and V_space(elem) not in VectorSpace(Fq, m).subspace([V_space(b) for b in true_support]):
        true_support.append(elem)


mock_error_coords = []
for j in range(n):
    coord_element = sum(Fq(random.randint(0, q-1)) * true_support[i] for i in range(r))
    mock_error_coords.append(coord_element)
e_secret = vector(Fqm, mock_error_coords)

# Calculate syndrome vector s
s_vec = H_mat * e_secret

E_matrix = matrix(Fq, [V_space(el) for el in e_secret]).transpose()

print("--- PROBLEM INSTANCE ---")
print(f"Secret Target Vector e :\n{e_secret}\n")
print(f"Base Field Matrix E ({E_matrix.nrows()}x{E_matrix.ncols()}):\n{E_matrix}\n")
print(f"Syndrome Vector s      : {s_vec}\n")

num_erasures = 2  
J_cols = sorted(random.sample(range(n), num_erasures))
a_J_vals = [e_secret[j] for j in J_cols]

print(f"Randomly Leaked Column Indices J : {J_cols}")
print(f"Leaked Column Field Elements a_J : {a_J_vals}\n")

recovered_solutions = full_entry_residual_reduction_module(
    q=q, m=m, n=n, k=k, r=r, J=J_cols, a_J=a_J_vals, H=H_mat, s=s_vec, B_rec=128
)

print(f"Total verified candidates found: {len(recovered_solutions)}")

found_secret = False
for index, candidate in enumerate(recovered_solutions):
    if candidate == e_secret:
        print(f"[SUCCESS] Candidate index {index} matches the true secret e_secret exactly!")
        print(f"  Recovered: {candidate}")
        found_secret = True
        break

if not found_secret:
    print("[-] Secret vector was not recovered in this reduction window.")

--- PROBLEM INSTANCE ---
Secret Target Vector e :
(z^3 + z, z^3 + z, z^3 + z^2 + z, 0, z^3 + z, z^3 + z^2 + z, 0)

Base Field Matrix E (4x7):
[0 0 0 0 0 0 0]
[1 1 1 0 1 1 0]
[0 0 1 0 0 1 0]
[1 1 1 0 1 1 0]

Syndrome Vector s      : (z^3 + z, 0, z^3 + 1, z^2 + 1)

Randomly Leaked Column Indices J : [2, 4]
Leaked Column Field Elements a_J : [z^3 + z^2 + z, z^3 + z]

[+] Algorithm 4: Evaluated erasure subspace dimension rho_J = 2
[+] Algorithm 4: Bounds calculated: tau_min = 0, tau_max = 2
[+] Step 9: Returning candidate pool E_entry containing 1 solutions.
Total verified candidates found: 1
[SUCCESS] Candidate index 0 matches the true secret e_secret exactly!
  Recovered: (z^3 + z, z^3 + z, z^3 + z^2 + z, 0, z^3 + z, z^3 + z^2 + z, 0)
